# Bronze Pipeline Shared Helpers

Lightweight widget, JSON and table helpers. No Spark cache or persistence is used.

In [0]:
import json
import uuid
from datetime import datetime, timezone

_BRONZE_WIDGET_DEFAULTS = {
    "pipeline_run_id": "",
    "force_full_refresh": "false",
    "create_cutover_backups": "true",
    "run_post_deployment_checks": "false",
    "run_map_pipeline": "true",
    "run_device_pipeline": "true",
}

for _bronze_widget_name, _bronze_widget_default in _BRONZE_WIDGET_DEFAULTS.items():
    try:
        dbutils.widgets.text(_bronze_widget_name, _bronze_widget_default)
    except Exception:
        pass

def bronze_value(name: str, default: str = "") -> str:
    try:
        value = dbutils.widgets.get(name)
        return default if value is None else str(value).strip()
    except Exception:
        return default

def bronze_bool(name: str, default: bool = False) -> bool:
    return bronze_value(name, str(default).lower()).lower() in {"1", "true", "yes", "y"}

def bronze_table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def bronze_json(value) -> str:
    return json.dumps(value, default=str, sort_keys=True)

def bronze_run_id() -> str:
    requested = bronze_value("pipeline_run_id", "")
    return requested or str(uuid.uuid4())

def bronze_utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()